<a href="https://colab.research.google.com/github/RamiAmasha31/ABM-applied_mathmatics/blob/main/After_recovery_factors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

class Agent:
    def __init__(self, num_strains, infection_params, immunity_params,infection_probabilities):
        self.infected = [False] * num_strains  # List to track infection status for each strain
        self.immune = [False] * num_strains  # List to track immunity status for each strain
        self.remaining_days_of_infection = [self.random_days(mean, std) for mean, std in infection_params]
        self.remaining_days_of_immunity = [self.random_days(mean, std) for mean, std in immunity_params]
        self.susceptible = True
        self.infection_probabilities = infection_probabilities.copy()  # Copy to ensure independence
    def random_days(self, mean, std):
        return max(0, int(np.random.normal(mean, std)))


In [2]:
import random
def PickTwoAgentsWithContacts(numAgents):
    """
    Randomly selects two different agent indices from the range [0, numAgents-1]
    with a probability of being in contact.

    Parameters:
    - numAgents (int): The total number of agents.
    - contact_prob (float): The probability of two agents being in contact.

    Returns:
    - tuple: A tuple containing two distinct agent indices.
    """
    # Randomly select two initial indices
    index1 = random.randint(0, numAgents - 1)
    index2 = random.randint(0, numAgents - 1)

    # Ensure the selected indices are different
    while index1 == index2 :
        index1 = random.randint(0, numAgents - 1)
        index2 = random.randint(0, numAgents - 1)

    return index1, index2

In [3]:
import numpy as np
import random


def start_step(number_of_agents, num_strains, infected_at_start, infection_params, immunity_params,infection_probabilities):
    if sum(infected_at_start) > number_of_agents:
        raise ValueError("Total number of initially infected agents exceeds the total number of agents.")

    agents = []

    # Create agents
    for i in range(number_of_agents):
        agents.append(Agent(num_strains, infection_params, immunity_params,infection_probabilities))

    # Infect agents according to infected_at_start array
    for strain_index in range(num_strains):
        infected_indices = random.sample(range(number_of_agents), infected_at_start[strain_index])
        for index in infected_indices:
            agents[index].infected[strain_index] = True
            agents[index].remaining_days_of_infection[strain_index] = max(0, int(np.random.normal(infection_params[strain_index][0], infection_params[strain_index][1])))
            agents[index].susceptible = False

    return agents


In [4]:
import random
import numpy as np

def infect(agent1, agent2, infection_probabilities,infection_params):
    for strain_index in range(len(infection_probabilities)):
        random_number = random.random()
        if not agent1.infected[strain_index] and agent2.infected[strain_index] and not agent1.immune[strain_index] and random_number < agent1.infection_probabilities[strain_index]:
            agent1.infected[strain_index] = True
            agent1.remaining_days_of_infection[strain_index] =max(0, int(np.random.normal(infection_params[strain_index][0], infection_params[strain_index][1])))
            agent1.susceptible = False
        elif not agent2.infected[strain_index] and agent1.infected[strain_index] and not agent2.immune[strain_index] and random_number < agent2.infection_probabilities[strain_index]:
            agent2.infected[strain_index] = True
            agent2.remaining_days_of_infection[strain_index] = max(0, int(np.random.normal(infection_params[strain_index][0], infection_params[strain_index][1])))
            agent2.susceptible = False

    return agent1, agent2


In [5]:
import numpy as np
def remove_dead_agents(agents, num_dead_agents):
    if num_dead_agents >= len(agents):
        agents.clear()
    else:
        for _ in range(num_dead_agents):
            index = random.randint(0, len(agents) - 1)
            agents.pop(index)
    return agents

def create_new_agents(agents, num_new_agents, infected_per_birth_duration, infection_params, immunity_params, infection_probabilities,birth_duration):
    num_strains = len(infection_params)  # Assuming all agents have the same number of strains

    if len(infected_per_birth_duration) != num_strains:
        raise ValueError("Length of infected_per_birth_duration must match the number of strains.")

    for i in range(num_new_agents):
        new_agent = Agent(num_strains, infection_params, immunity_params, infection_probabilities)

        for strain_index in range(num_strains):
            number=np.random.poisson(infected_per_birth_duration[strain_index]/birth_duration)
            if i < number:
                new_agent.infected[strain_index] = True

        agents.append(new_agent)

    return agents


In [6]:
import numpy as np

def Birth_death(agents, birth_pulse, birth_rate_yearly, birth_interval, birth_duration, new_born, deads,carrying_capacity,infection_probabilities, infected_per_birth_duration, infection_params, immunity_params):
    num_new_agents = 0
    num_dead_agents = 0
    b = 0.1
    m=(birth_rate_yearly-b)/int(len(agents))



    # Check if the birth pulse matches the birth interval
    if birth_pulse % birth_interval < birth_duration:
        # Calculate the expected number of new agents over the birth duration
        expected_new_agents_over_duration = birth_rate_yearly * len(agents)

        # Generate the number of new agents using Poisson distribution
        num_new_agents = np.random.poisson(expected_new_agents_over_duration/birth_duration)

        # Create new agents with the specified parameters
        agents = create_new_agents(agents, num_new_agents, infected_per_birth_duration, infection_params, immunity_params, infection_probabilities,birth_duration)

    # Calculate daily death rate
    death_rate = m * len(agents) + b

    daily_death_rate=death_rate/birth_interval


    # Generate the number of dead agents using binomial distribution
    num_dead_agents = np.random.binomial(len(agents), daily_death_rate)

    # Remove dead agents
    agents = remove_dead_agents(agents, num_dead_agents)

    return agents, num_new_agents, num_dead_agents

In [7]:
import pandas as pd
import numpy as np

def update_params(agents,immunity_params,after_recovery_factors):
    num_strains = len(agents[0].infected)  # Assuming all agents have the same number of strains
    Overall_infected = [0] * num_strains  # List to track infected agents for each strain
    susceptibles = 0

    # List to collect debug information
    debug_info = []

    for agent_idx, agent in enumerate(agents):
        agent_debug_info = {
            "Agent": agent_idx,
            "Strains Infected": [],
            "Strains Immune": [],
            "Infection Probabilities Before": [],
            "Infection Probabilities After": []
        }

        for strain_index in range(num_strains):
            if agent.infected[strain_index]:
                agent.remaining_days_of_infection[strain_index] -= 1

                if agent.remaining_days_of_infection[strain_index] < 0:
                    # Infection duration ended
                    before_probs = agent.infection_probabilities.copy()
                    agent.infected[strain_index] = False
                    agent.immune[strain_index] = True
                    ## change to exp
                   # agent.remaining_days_of_immunity[strain_index] =np.random.exponential(180, 1)

                    agent.remaining_days_of_immunity[strain_index] = np.random.normal(immunity_params[strain_index][0], immunity_params[strain_index][1])

                    # Update infection probabilities only for the current agent
                    for idx in range(num_strains):
                        if idx != strain_index:
                            agent.infection_probabilities[idx] *= after_recovery_factors[strain_index]


            elif agent.immune[strain_index]:
                agent.remaining_days_of_immunity[strain_index] -= 1

                if agent.remaining_days_of_immunity[strain_index] < 0:
                    # Immunity duration ended
                    before_probs = agent.infection_probabilities.copy()
                    agent.immune[strain_index] = False
                    agent.susceptible = True

                    # Update infection probabilities only for the current agent
                    for idx in range(num_strains):
                        if idx != strain_index:
                              agent.infection_probabilities[idx] /= after_recovery_factors[strain_index]



        for strain_index in range(num_strains):
            if agent.infected[strain_index]:
                Overall_infected[strain_index] += 1
        if agent.susceptible:
            susceptibles += 1



    return agents, Overall_infected, susceptibles


In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.fftpack
# from scipy.signal import find_peaks

%matplotlib inline

def random_days1(mean, std):
    return max(0, int(np.random.normal(mean, std)))
def run_simulation(num_agents = 1000,num_strains = 1,infected_at_start = [5],infection_params = [(60, 20)],
immunity_params = [(180, 90)],infected_per_birth_duration = [1],after_recovery_factors = [1],R0 = [4],
                   simulation_duration1 = 20000,birth_rate_yearly = 0.3,birth_duration = 365,birth_interval = 365,carrying_capacity = 1000):

  number_of_interactions_per_day_per_agent = 1 / 10
  infection_probabilities = [0] * num_strains
  birth_pulse = 0
  new_born = [0] * int(simulation_duration1)
  deads = [0] * int(simulation_duration1)
  population = [0] * int(simulation_duration1)
  overAll_infected = [[] for _ in range(num_strains)]
  overAll_infected_transform = [[] for _ in range(num_strains)]
  overall_infected_combined = []
  intersections = []
  sus = [0] * int(simulation_duration1)
  number_of_interactions_per_day = int(num_agents * number_of_interactions_per_day_per_agent // 2)

  # Track the infection probabilities for a specific agent
  specific_agent_infection_probs = []
  pop=[0]*simulation_duration1
  # Loop through each strain's infection parameters
  for idx, (mean, std) in enumerate(infection_params):
      TMP = mean * number_of_interactions_per_day_per_agent
      infection_probabilities[idx] = (float(R0[idx]) / TMP)
      #print(f"Infection Probability for Strain {idx}: {infection_probabilities[idx]}")




  agents = start_step(num_agents, num_strains, infected_at_start, infection_params, immunity_params, infection_probabilities)


  for i in range(simulation_duration1):
      for j in range(number_of_interactions_per_day):
          index1, index2 = PickTwoAgentsWithContacts(len(agents))
          infect(agents[index1], agents[index2], infection_probabilities,infection_params)

      birth_pulse += 1
      agents, num_new_agents, num_dead_agents = Birth_death(agents, birth_pulse, birth_rate_yearly,
                                                            birth_interval, birth_duration,
                                                            new_born, deads, carrying_capacity,
                                                            infection_probabilities, infected_per_birth_duration, infection_params, immunity_params)

      deads[i] = num_dead_agents
      new_born[i] = num_new_agents
      population[i] = len(agents)

      # Update agents and get current infected counts for each strain
      agents, overall_infected_current, susceptibles = update_params(agents,immunity_params,after_recovery_factors)
      # print(agents[0].infection_probabilities)


      # Accumulate infected counts for each strain
      for strain_index in range(num_strains):
          overAll_infected[strain_index].append(overall_infected_current[strain_index])

      # Calculate overall infected agents (infected in at least one strain)
      overall_infected_count = sum(1 for agent in agents if any(agent.infected))
      overall_infected_combined.append(overall_infected_count)
  return overAll_infected

  # # Plotting each strain on the same plot with different colors
  # plt.figure(figsize=(12, 8))
  # colors = ['b', 'g', 'r']  # Different colors for different strains

  # for strain_index in range(num_strains):
  #   # overAll_infected_transform[strain_index]=scipy.fftpack.fft(overAll_infected[strain_index])
  #     plt.plot(range(simulation_duration1), overAll_infected[strain_index], color=colors[strain_index], label=f'Strain {strain_index + 1}')

  # plt.xlabel('Time (days)')
  # plt.ylabel('Infected Agents')
  # plt.title('Infected Agents Over Time by Strain')
  # plt.legend()
  # plt.grid(True)
  # plt.tight_layout()
  # plt.show()

OVER_ALL=run_simulation(num_agents = 1000,num_strains = 1,infected_at_start = [5],infection_params = [(60, 20)],
immunity_params = [(180, 90)],infected_per_birth_duration = [1],after_recovery_factors = [1],R0 = [4],
                   simulation_duration1 = 6000,birth_rate_yearly = 0.3,birth_duration = 365,birth_interval = 365,carrying_capacity = 1000)

In [9]:
!pip install openpyxl
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 3.9 MB/s eta 0:00:00


Here we call the main simulation function. You may change the simulation duration and add the list of r0 values you want to iterate on.

#The following code for running different set of values for after_recovery_factors

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from docx import Document
import seaborn as sns  # Import seaborn for heatmaps
from docx.shared import Inches

# Document to store results
doc = Document()

# Add a detailed description of the calculations
doc.add_heading('Simulation Results', level=1)
doc.add_paragraph(
    "This document presents the results of the simulation conducted to analyze the infection dynamics of multiple strains. "
    "The main goal is to observe how different after-recovery factors influence the number of infected agents over time "
    "and the correlation between different strains.\n\n"
    "The simulation involves the following key steps:\n"
    "1. **Simulation Execution**: For each set of after-recovery factors (with values ranging from 0.1 to 1.0 in increments of 0.1), "
    "the `run_simulation` function is called to model the infection dynamics for three different strains. The parameters include "
    "the number of agents, initial infections, infection rates, immunity duration, and more.\n"
    "2. **Infected Agents Tracking**: The simulation tracks the number of infected agents for each strain over the specified "
    "duration.\n"
    "3. **Correlation Calculation**: After each simulation run, the correlation matrix is computed to analyze the relationship "
    "between the infected counts of the different strains. The Pearson correlation coefficient is calculated between each pair of strains.\n"
    "4. **Overall Correlation**: The overall correlation for each set of after-recovery factors is determined by calculating the "
    "mean of the Pearson correlation coefficients obtained from the previous step. This value gives an indication of how well "
    "the infection dynamics of the strains correlate with each other for the given recovery factors.\n"
    "5. **Visualization**: The results are visualized through line plots representing the infected counts over time and heatmaps "
    "showing the correlation between strains. Finally, all results are saved in a Word document for further analysis."
)

# Generate a list of after_recovery_factors with three equal values from 0.1 to 1.0 in steps of 0.1
after_recovery_factors_list = [[round(i, 1)] * 3 for i in np.arange(0.1, 1.1, 0.1)]  # Format to one decimal

# Prepare to store overall correlation results
overall_correlation_results = []

# Loop through each set of after_recovery_factors
for after_recovery_factors in after_recovery_factors_list:
    # Call the existing run_simulation function
    over_all_infected = run_simulation(
        num_agents=1000,
        num_strains=3,
        infected_at_start=[5, 5, 5],
        infection_params=[(60, 20), (60, 20), (60, 20)],
        immunity_params=[(180, 30), (180, 30), (180, 30)],
        infected_per_birth_duration=[1, 1, 1],
        after_recovery_factors=after_recovery_factors,
        R0=[3, 4, 5],
        simulation_duration1=20000,  # Set simulation duration to 20000
        birth_rate_yearly=0.3,
        birth_duration=365,
        birth_interval=365,
        carrying_capacity=1000
    )

    # Consider only days from 3000 to 20000
    time_range = range(3000, 20001)  # Days from 3000 to 20000
    filtered_infected = [infected[3000:20001] for infected in over_all_infected]

    # Calculate the correlation matrix for infected agents
    correlation_matrix = np.corrcoef(filtered_infected)

    # Calculate significance for the correlation results
    correlation_significance = []
    for i in range(len(correlation_matrix)):
        for j in range(i + 1, len(correlation_matrix)):
            corr, _ = pearsonr(filtered_infected[i], filtered_infected[j])
            correlation_significance.append((after_recovery_factors, corr))

    # Calculate the overall correlation for this set
    overall_correlation = np.mean([corr for _, corr in correlation_significance])
    overall_correlation_results.append((after_recovery_factors, overall_correlation))

    # Plotting the infected agents for each strain
    plt.figure(figsize=(12, 6))
    for strain_index in range(len(filtered_infected)):
        plt.plot(time_range, filtered_infected[strain_index], label=f'Strain {strain_index + 1}')

    plt.xlabel('Time (days)')
    plt.ylabel('Infected Agents')
    plt.title(f'Infected Agents Over Time (after_recovery_factors={after_recovery_factors})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'infected_agents_after_recovery_factors_{after_recovery_factors}.png')  # Save the plot
    plt.close()  # Close the plot to avoid displaying it in Jupyter

    # Add plot to document with description
    doc.add_heading(f'Infected Agents Plot (after_recovery_factors={after_recovery_factors})', level=2)
    doc.add_picture(f'infected_agents_after_recovery_factors_{after_recovery_factors}.png', width=Inches(6.0))
    doc.add_paragraph("This plot shows the number of infected agents for each strain over time with different recovery factors.")

    # Create heatmap for the correlation matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8},
                xticklabels=[f'Strain {i + 1}' for i in range(len(filtered_infected))],
                yticklabels=[f'Strain {i + 1}' for i in range(len(filtered_infected))])

    plt.title(f'Correlation Heatmap (after_recovery_factors={after_recovery_factors})')
    plt.tight_layout()
    plt.savefig(f'correlation_heatmap_after_recovery_factors_{after_recovery_factors}.png')  # Save the heatmap
    plt.close()  # Close the heatmap plot

    # Add heatmap to document with description
    doc.add_heading(f'Correlation Heatmap (after_recovery_factors={after_recovery_factors})', level=2)
    doc.add_picture(f'correlation_heatmap_after_recovery_factors_{after_recovery_factors}.png', width=Inches(6.0))
    doc.add_paragraph("This heatmap illustrates the correlation between the infected agents of different strains.")

# Create a table for the sets organized by their overall correlation
doc.add_heading('Overall Correlation Results', level=1)
overall_table = doc.add_table(rows=1, cols=2)
overall_table.cell(0, 0).text = 'After Recovery Factors'
overall_table.cell(0, 1).text = 'Overall Correlation'

# Sort overall correlation results
sorted_overall_correlation_results = sorted(overall_correlation_results, key=lambda x: abs(x[1]), reverse=True)

# Populate the overall correlation table
for result in sorted_overall_correlation_results:
    after_recovery_factors, overall_corr = result
    row_cells = overall_table.add_row().cells
    row_cells[0].text = str(after_recovery_factors)
    row_cells[1].text = f'{overall_corr:.2f}'  # Display with two decimal places

# Save the document
doc.save('Simulation_Results.docx')

print("Simulation completed, results saved to 'Simulation_Results.docx'.")
